In [1]:
import pandas as pd
import numpy as np

# 재현성 고정
np.random.seed(42)

# 데이터 로드
orders = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\orders.csv")

log = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\production_log.csv")

defect = pd.read_csv(r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\00_used_data\manufacturing_defect_dataset.csv")

print("orders:\n", orders.head(), "\n")
print("log:\n", log.head(), "\n")
print("defect:\n", defect.head(), "\n")
print("defect columns:", defect.columns.tolist())

orders:
     order_id product_id  order_qty  order_date    due_date
0  ORD000001     PRD012         35  2025-11-19  2025-11-27
1  ORD000002     PRD002         87  2025-10-17  2025-11-04
2  ORD000003     PRD004         50  2025-08-28  2025-09-08
3  ORD000004     PRD008         65  2025-09-23  2025-10-11
4  ORD000005     PRD009         37  2025-11-02  2025-11-16 

log:
       lot_id process_id   order_id product_id machine_id  actual_work_time  \
0  LOT000001          A  ORD000001     PRD012    MILL_04            103.70   
1  LOT000001          B  ORD000001     PRD012   LATHE_02             87.66   
2  LOT000001          C  ORD000001     PRD012   DRILL_01             91.34   
3  LOT000001          D  ORD000001     PRD012   GRIND_01             87.25   
4  LOT000001          E  ORD000001     PRD012     ADD_01            115.64   

   setup_time  downtime job_status  
0       22.84      2.78    Delayed  
1       10.26      3.75  Completed  
2        9.07      8.60  Completed  
3        8.4

In [2]:
# 필요한 컬럼만 추출
defect_base = defect[[
    "ProductionVolume",
    "ProductionCost",
    "DefectRate",
    "QualityScore",
    "MaintenanceHours",
    "DowntimePercentage"
]].copy()

# 결측 제거
defect_base = defect_base.dropna().reset_index(drop=True)

In [3]:
# 주문 수량 매핑
order_qty_map = orders.set_index("order_id")["order_qty"]

In [4]:
# 공정별 민감도 설정
process_defect_factor = {
    "A" : 1.00,
    "B" : 1.05,
    "C" : 0.90,
    "D" : 1.20,
    "E" : 1.15
}

In [13]:
# 데이터 적재
rows = []

for _, row in log.iterrows():
    order_id = row["order_id"]
    process_id = row["process_id"]

    # 주문 수량을 공정 투입 수량으로 사용
    input_qty = int(order_qty_map[order_id])

    # Defect Dataset에서 1행 샘플링
    sampled = defect_base.sample(n=1).iloc[0]

    # 실제 데이터 기반 값
    sampled_defect = np.random.choice(defect["DefectRate"])
    sampled_defect_rate = sampled_defect / 35
    sampled_quality_score = sampled["QualityScore"]
    sampled_production_cost = sampled["ProductionCost"]
    sampled_maintenance_hours = sampled["MaintenanceHours"]
    sampled_downtime_percentage = sampled["DowntimePercentage"]

    # downtime 영향 반영
    downtime = row["downtime"]

    if downtime < 10:
        downtime_factor = 1.00
    elif downtime < 30:
        downtime_factor = 1.15
    else:
        downtime_factor = 1.35
    
    # 공정별 민감도 반영
    process_factor = process_defect_factor[process_id]

    # 최종 실제 불량률
    actual_defect_rate = (
        sampled_defect_rate * downtime_factor * process_factor * np.random.uniform(0.95, 1.05)
    )

    # 비정상적으로 커지는 값 방지
    actual_defect_rate = np.clip(actual_defect_rate, 0.005, 0.25)
    actual_defect_rate = round(actual_defect_rate, 4)

    # 수량 계산
    defect_qty = int(round(input_qty * actual_defect_rate))
    output_qty = input_qty - defect_qty

    # 품질 점수
    quality_score = (
        sampled_quality_score - (actual_defect_rate * 100 * 0.3) - (downtime * 0.03)
    )

    quality_score = round(np.clip(quality_score, 0, 100), 2)

    # 생산 비용
    production_cost = (
        sampled_production_cost * (input_qty / max(sampled["ProductionVolume"], 1)) + downtime * 2
    )

    production_cost = round(production_cost, 2)

    # 유지보수 시간 / 다운타임 비율
    maintenance_hours = round(
        sampled_maintenance_hours + (downtime * 0.05), 2
    )

    downtime_percentage = round(
        max(
            sampled_downtime_percentage, row["downtime"] / row["actual_work_time"]
        ), 4
    )

    # 결과 상태
    if actual_defect_rate >= 0.15:
        result_status = "Fail"
    elif actual_defect_rate >= 0.08:
        result_status = "Warning"
    else:
        result_status = "Normal"
    
    rows.append({
        "lot_id" : row["lot_id"],
        "process_id" : process_id,
        "order_id" : order_id,
        "product_id" : row["product_id"],
        "input_qty" : input_qty,
        "output_qty" : output_qty,
        "defect_qty" : defect_qty,
        "actual_defect_rate" : actual_defect_rate,
        "quality_score" : quality_score,
        "production_cost" : production_cost,
        "maintenance_hours" : maintenance_hours,
        "downtime_percentage" : downtime_percentage,
        "result_status" : result_status
    })

raw_production_result = pd.DataFrame(rows)

In [14]:
# ============================================================
# raw_production_result QC
# ============================================================

print("===== 기본 정보 =====")
print("row 수:", len(raw_production_result))
print("columns:", raw_production_result.columns.tolist())

print("\n===== PK 체크 =====")
print(
    "PK 중복:",
    raw_production_result.duplicated(["lot_id", "process_id"]).sum()
)

print("\n===== FK 체크 =====")
print(
    "lot_id FK 정상 여부:",
    raw_production_result["lot_id"].isin(log["lot_id"]).all()
)

print(
    "order_id FK 정상 여부:",
    raw_production_result["order_id"].isin(orders["order_id"]).all()
)

print("\n===== 수량 체크 =====")
print("output_qty 음수:", (raw_production_result["output_qty"] < 0).sum())
print("defect_qty 음수:", (raw_production_result["defect_qty"] < 0).sum())
print(
    "수량 정합성 오류:",
    (
        raw_production_result["input_qty"]
        != raw_production_result["output_qty"] + raw_production_result["defect_qty"]
    ).sum()
)

print("\n===== 불량률 분포 =====")
print(raw_production_result["actual_defect_rate"].describe())

print("\n===== 품질 점수 분포 =====")
print(raw_production_result["quality_score"].describe())

print("\n===== 결과 상태 분포 =====")
print(raw_production_result["result_status"].value_counts(normalize=True))

print("\n===== 공정별 평균 불량률 =====")
print(
    raw_production_result
    .groupby("process_id")["actual_defect_rate"]
    .mean()
    .sort_index()
)

===== 기본 정보 =====
row 수: 25000
columns: ['lot_id', 'process_id', 'order_id', 'product_id', 'input_qty', 'output_qty', 'defect_qty', 'actual_defect_rate', 'quality_score', 'production_cost', 'maintenance_hours', 'downtime_percentage', 'result_status']

===== PK 체크 =====
PK 중복: 0

===== FK 체크 =====
lot_id FK 정상 여부: True
order_id FK 정상 여부: True

===== 수량 체크 =====
output_qty 음수: 0
defect_qty 음수: 0
수량 정합성 오류: 0

===== 불량률 분포 =====
count    25000.000000
mean         0.086720
std          0.043453
min          0.012500
25%          0.049500
50%          0.084900
75%          0.120300
max          0.238400
Name: actual_defect_rate, dtype: float64

===== 품질 점수 분포 =====
count    25000.000000
mean        77.182236
std         11.723040
min         52.920000
25%         66.947500
50%         77.220000
75%         87.390000
max         99.430000
Name: quality_score, dtype: float64

===== 결과 상태 분포 =====
result_status
Normal     0.46716
Warning    0.45280
Fail       0.08004
Name: proportion, dtype: f

In [15]:
# CSV로 내보내기
output_path = r"C:\Users\jydom\OneDrive\문서\Project\manufacturing\00_data\01_raw_data\production_result.csv"

try:
    raw_production_result.to_csv(output_path, index=False, encoding='utf-8-sig')
    print("process.csv 생성 완료:", raw_production_result.shape)
    
except Exception as e:
    print("생성 실패:", e)

process.csv 생성 완료: (25000, 13)
